In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

current_working_directory = os.getcwd()
print(f"Current Working Directory: {current_working_directory}\n")
python_paths = sys.path
print("Python's Search Paths (sys.path):")
for path in python_paths:
    print(f"- {path}")

def prepare():
    # module_path = os.path.abspath(os.path.join('..'))
    module_path = os.path.abspath(os.path.join('../', '../'))
    if module_path not in sys.path:
        sys.path.append(module_path)

print(sys.path)


In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_collective import run

In [ ]:
seeds = [0]
deltas = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

In [ ]:
model_params = dict(
    label = "SGC", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "linear",
    depth = 1,
    regularizer = 0.001,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license="/mnt/c/Users/emiel/gurobi.lic"

)

In [ ]:
data_params = dict(
    dataset = "cba",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        m = 2,
        # seed = 0 # used to generate the dataset & data split
    )
)

In [ ]:
import pandas as pd
import time

metrics_to_log = ["accuracy_test", "accuracy_trn", "accuracy_cert_pois_robust", "delta", "runtime"]
output_base_dir = "results/collective"
os.makedirs(output_base_dir, exist_ok=True)

dataset_name = "cba"

for delta in deltas:
    certificate_params["delta"] = delta
    summary_results = []
    print(f"  Running for delta: {delta:.2f}")

    for seed in seeds:
        data_params["specification"]["seed"] = seed
        start_time = time.time()
        result = run(data_params=data_params, model_params=model_params,
                     certificate_params=certificate_params, verbosity_params=verbosity_params,
                     other_params=other_params, seed=seed)
        runtime = round(time.time() - start_time, 2)
        result['runtime'] = runtime
        summary_results.append({k: result.get(k) for k in metrics_to_log})

    df = pd.DataFrame(summary_results, index=[f"Seed {s}" for s in seeds])
    df.index.name = "seed"
    output_filename = f"{dataset_name}-{delta:.2f}.csv"
    output_path = os.path.join(output_base_dir, output_filename)
    df.to_csv(output_path, index=True)

    print(f"  Results saved to: {output_path}")
    print(df)
    print("-" * 30)


In [ ]:

#  SGC on BOTH CBA Datasets - Complexity Cascade 
import os
import sys
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.cuda.is_available = lambda: False
if torch.cuda.is_available():
    torch.cuda.set_device('cpu')

module_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added '{module_path}' to sys.path to find project modules.")

from exp_labelcert_collective import run

model_params = {
    'label': 'SGC',
    'model': 'GCN',  
    'normalization': 'sym_normalization',  
    'activation': 'linear',  
    'depth': 1,
    'regularizer': 0.001,
    'pred_method': 'svm',
    'bias': False,
    'alpha_tol': 1e-4,
    'solver': 'qplayer'
}

base_data_params = {
    'learning_setting': 'transductive',
    'specification': {
        'classes': 2,
        'n_trn_labeled': 10,
        'n_trn_unlabeled': 0,
        'n_val': 10,
        'n_test': 180,
        'sigma': 1,
        'avg_within_class_degree': 3.16,  
        'avg_between_class_degree': 0.74,  
        'm': 2,  
        'K': 1.5  
    }
}

cba_configs = {
    'cba': {
        'dataset': 'cba',
        **base_data_params
    },
    'cba_sparsity_1': {
        'dataset': 'cba',
        **base_data_params,
        'specification': {
            **base_data_params['specification'],
            'm': 4  
        }
    },
    'cba_sparsity_2': {
        'dataset': 'cba',
        **base_data_params,
        'specification': {
            **base_data_params['specification'],
            'm': 1  
        }
    }
}

other_params = {
    'device': 'cpu',  
    'dtype': torch.float64,
    'allow_tf32': False,
    'path_gurobi_license': '/mnt/c/Users/emiel/gurobi.lic'  
}

certificate_base_params = {
    'TimeLimit': 86400,
    'LogToConsole': 1,
    'OutputFlag': 1,
    'Threads': 2,
    'Presolve': 2
}

verbosity_params = {'debug_lvl': 'warning'}



#  grid is dense around the critical region to capture the cliff
eps_coarse = np.linspace(0.00, 0.30, 16).tolist()
eps_fine = np.linspace(0.13, 0.18, 26).tolist()
epsilons = sorted(set(eps_coarse + eps_fine))

additional_eps = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 1.0]
epsilons = sorted(set(epsilons + additional_eps))

seeds = range(10)  

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
output_filename = "final_sgc_on_cba_collective_data.csv"
output_path = log_dir / output_filename


print("--- Starting Final Data Generation for SGC on CBA (Collective) ---")
print(f"Running {len(epsilons)} epsilon points across {len(seeds)} seeds...")
print(f"Total experiments: {len(epsilons) * len(seeds)}")
all_results = []
start_time_total = time.time()

for seed_val in seeds:
    print(f"\nProcessing Seed: {seed_val}/{len(seeds)-1}...")
    data_params['specification']['seed'] = int(seed_val)
    
    for i, eps in enumerate(epsilons):
        certificate_params = certificate_base_params.copy()
        certificate_params['delta'] = float(eps)
        
        try:
            out = run(
                data_params=data_params,
                model_params=model_params,
                certificate_params=certificate_params,
                verbosity_params=verbosity_params,
                other_params=other_params,
                seed=int(seed_val)
            )
            
            out.update({
                'delta': float(eps), 
                'seed': int(seed_val),
                'dataset': 'cba',
                'model': 'SGC'
            })
            all_results.append(out)
            
            robust_ratio = out.get('accuracy_cert_pois_robust', 'N/A')
            gurobi_nodes = out.get('gurobi_node_count', 'N/A')
            print(f"  ({i+1}/{len(epsilons)}) ε={eps:.4f} | Robust Ratio: {robust_ratio} | Gurobi Nodes: {gurobi_nodes}")
            
        except Exception as e:
            print(f"  ERROR at ε={eps:.4f}, seed={seed_val}: {str(e)}")
            error_result = {
                'delta': float(eps),
                'seed': int(seed_val),
                'dataset': 'cba',
                'model': 'SGC',
                'error': str(e),
                'accuracy_cert_pois_robust': np.nan,
                'gurobi_node_count': np.nan
            }
            all_results.append(error_result)

df_final = pd.DataFrame(all_results)
df_final.to_csv(output_path, index=False)

end_time_total = time.time()
print(f"\n--- ✅ Experiment Complete ---")
print(f"Total runtime: {(end_time_total - start_time_total) / 60:.2f} minutes")
print(f"Final data for {len(df_final)} runs saved to '{output_path}'")
print(f"Success rate: {len(df_final[~df_final.get('error', pd.Series()).notna()])} / {len(df_final)}")

if len(df_final) > 0:
    print(f"\n--- Quick Summary ---")
    valid_results = df_final[~df_final.get('error', pd.Series()).notna()]
    if len(valid_results) > 0:
        print(f"Valid results: {len(valid_results)}")
        
        for config_name in cba_configs.keys():
            config_results = valid_results[valid_results['dataset'] == config_name]
            if len(config_results) > 0:
                print(f"\n{config_name}:")
                if 'accuracy_cert_pois_robust' in config_results.columns:
                    print(f"  Robust ratio range: {config_results['accuracy_cert_pois_robust'].min():.4f} - {config_results['accuracy_cert_pois_robust'].max():.4f}")
                if 'gurobi_node_count' in config_results.columns:
                    print(f"  Gurobi nodes range: {config_results['gurobi_node_count'].min()} - {config_results['gurobi_node_count'].max()}")